In [1]:
from datetime import date, timedelta

import polars as pl

from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from data_engineering.flights.flight_type import get_flights


start_date = date(2026, 5, 14)
end_date = date(2026, 5, 27)
target_dates = [
    start_date + timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
]

df_flights = pl.concat([get_flights(target_date) for target_date in target_dates])
df_ladd_flights = df_flights.filter(pl.col("ladd").fill_null(False))

ladd_icaos = df_ladd_flights.get_column("icao").drop_nulls().unique().sort().to_list()
df_adsb = pl.concat([
    read_icaos_from_adsb(target_date, ladd_icaos)
    for target_date in target_dates
])

{
    "start_date": start_date,
    "end_date": end_date,
    "days": len(target_dates),
    "ladd_flights": len(df_ladd_flights),
    "ladd_icaos": len(ladd_icaos),
    "adsb_rows": len(df_adsb),
}


{'start_date': datetime.date(2026, 5, 14),
 'end_date': datetime.date(2026, 5, 27),
 'days': 14,
 'ladd_flights': 110365,
 'ladd_icaos': 12024,
 'adsb_rows': 58113550}

In [2]:
from flights.flights_match_adsb import flights_match_adsb


match_results = []
for icao in ladd_icaos:
    icao_flights = df_ladd_flights.filter(pl.col("icao") == icao).sort("takeoff_time")
    icao_adsb = df_adsb.filter(pl.col("icao") == icao).sort("time")

    matches = flights_match_adsb(icao_flights, icao_adsb)

    match_results.append({
        "icao": icao,
        "matches": matches,
        "flight_rows": len(icao_flights),
        "adsb_rows": len(icao_adsb),
    })

df_match_results = pl.DataFrame(match_results)
match_counts = df_match_results.group_by("matches").len().sort("matches", descending=True)
true_matches = df_match_results.filter(pl.col("matches")).height
total_icaos = df_match_results.height

{
    "true_matches": true_matches,
    "false_matches": df_match_results.filter(~pl.col("matches")).height,
    "total_icaos": total_icaos,
    "true_match_percentage": true_matches / total_icaos * 100,
}

{'true_matches': 3488,
 'false_matches': 8536,
 'total_icaos': 12024,
 'true_match_percentage': 29.008649367930804}

In [3]:
matched_icaos = df_match_results.filter(pl.col("matches")).get_column("icao").to_list()
df_ladd_flights_matched_adsb = df_ladd_flights.filter(pl.col("icao").is_in(matched_icaos))

{
    "matched_ladd_flights": len(df_ladd_flights_matched_adsb),
    "matched_ladd_icaos": len(matched_icaos),
}


{'matched_ladd_flights': 13927, 'matched_ladd_icaos': 3488}

In [9]:
df_ladd_flights_matched_adsb
path = "../../tmp/df_ladd_flights_matched_adsb.parquet"
df_ladd_flights_matched_adsb.write_parquet(path)

In [3]:
import polars as pl
path = "../../tmp/df_ladd_flights_matched_adsb.parquet"
df = pl.read_parquet(path)
df

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""300a00""","""SNM831""","""I-SEAF""",2026-05-14 14:41:29.900,"""LIRA""",2026-05-14 15:27:18.760,"""LIML""",false,true,false,false,"""GLF6""",null,"""GULFSTREAM G650""","""A3"""
"""300a00""","""SNM832""","""I-SEAF""",2026-05-14 16:29:22.030,"""LIML""",2026-05-14 17:14:39.420,"""LIRA""",false,true,false,false,"""GLF6""",null,"""GULFSTREAM G650""","""A3"""
"""3482d3""","""GES181M""","""EC-OJM""",2026-05-14 14:47:27.290,"""LECO""",2026-05-14 15:35:13.050,"""LEMD""",false,true,false,false,"""GA7C""",null,"""GULFSTREAM G700""","""A3"""
"""3c9c23""","""ADN95Y""","""D-BGAC""",2026-05-14 06:05:28.980,"""LIBD""",2026-05-14 07:51:44.490,"""EDDB""",false,true,false,false,"""CL35""",null,"""BOMBARDIER BD-100 Challenger 3…","""A2"""
"""3c9c23""","""ADN95Y""","""D-BGAC""",2026-05-14 09:01:43.700,"""EDDB""",2026-05-14 09:51:32.270,"""EDDM""",false,true,false,false,"""CL35""",null,"""BOMBARDIER BD-100 Challenger 3…","""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""c815b2""","""ZKHWI""","""ZK-HWI""",2026-05-27 01:43:09.360,"""NZTI""",2026-05-27 01:50:52.950,"""NZTI""",false,true,false,false,"""B06""",null,"""BELL 206 JetRanger""","""A7"""
"""e49505""","""PRGFT""","""PR-GFT""",2026-05-27 13:36:09.690,"""KBAF""",2026-05-27 14:47:03.930,"""KBAF""",false,true,false,false,"""GLF4""",null,"""GULFSTREAM 4""","""A2"""
"""e495ea""","""PRBDB""","""PR-BDB""",2026-05-27 20:40:08.340,"""SBJH""",2026-05-27 21:03:57.850,"""BR-0117""",false,true,false,false,"""E550""",null,"""EMBRAER EMB-550 Praetor 600""","""A2"""


In [5]:
from pia.simulator.main import main
df = main(df)
df.filter(pl.col("ladd") == True)

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a36588""","""N318LE""","""N318LE""",2026-05-27 11:59:05.830,"""KPTK""",2026-05-27 15:34:06.080,"""MMTO""",false,true,false,false,"""GA6C""","""BANC OF AMERICA LEASING & CAPI…","""GULFSTREAM G600""","""A3"""
"""48ba01""","""SPORB""","""SP-ORB""",2026-05-26 05:31:58.220,"""EPGL""",2026-05-26 06:44:55.820,"""EPPO""",false,true,false,false,"""SR22""",null,"""CIRRUS SR-22""","""A1"""
"""48ba01""","""SPORB""","""SP-ORB""",2026-05-27 15:59:48.170,"""EPPO""",2026-05-27 16:49:01.130,"""EPGL""",false,true,false,false,"""SR22""",null,"""CIRRUS SR-22""","""A1"""
"""781c91""","""B653A""","""B-653A""",2026-05-17 03:50:53.850,"""VHHH""",2026-05-17 04:11:55.500,"""ZGOW""",false,true,false,false,"""GLF6""",null,"""GULFSTREAM G650""","""A3"""
"""e49b6c""","""PRRZT""","""PR-RZT""",2026-05-27 11:45:48.810,"""SBRP""",2026-05-27 12:40:18,"""SBSP""",false,true,false,false,"""BE9L""",null,"""BEECH 90 King Air""","""A1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""43ea66""","""VSB07""","""M-CDJC""",2026-05-27 15:58:10.660,"""EGNL""",2026-05-27 16:45:50.060,"""EGGD""",false,true,false,false,"""BE20""",null,"""BEECH 200 Super King Air""","""A1"""
"""43ea66""","""VSB07""","""M-CDJC""",2026-05-27 17:31:42.210,"""EGGD""",2026-05-27 18:17:29.810,"""EGNL""",false,true,false,false,"""BE20""",null,"""BEECH 200 Super King Air""","""A1"""
"""a2c4db""","""N278GS""","""N278GS""",2026-05-18 18:15:37.990,"""KSJC""",2026-05-18 22:30:15.480,"""CYYZ""",false,true,false,false,"""F2TH""","""BANK OF UTAH TRUSTEE""","""DASSAULT Falcon 2000""","""A2"""


In [6]:
from data_engineering.flights.flight_type import flights_df_to_flights


flights = flights_df_to_flights(df)

In [7]:
from pia.evaluation.pia_graph_0 import flights_0, test_mapping_dict_0
from pia.evaluation.real_flights import flights_real_0
from pia.pia import build_graph, process_graph
print(f"{len(flights)} nodes")
pia_flights = [flight for flight in flights if flight.pia]
G = build_graph(flights)
# output_path = Path("tmp/pia_graph_0.png")
G, mapping_dict = process_graph(G)
print(f"{len(pia_flights)} pia nodes. {len(mapping_dict)} pia nodes identifed.")

13927 nodes
Removed 1340 nodes
Matched 1585
Matched 1495
11777 pia nodes. 3080 pia nodes identifed.
